In [1]:
import pandas as pd
import os

OFFICIAL_DATASET_PATH = "dataset_tesi_integrale.csv"
ATTACK_DATASET_PATH   = "dataset_attacco.csv"
OUTPUT_DATASET_PATH   = "dataset_tesi_etichettato.csv"

MATCHING_KEYS = ["Src_IP", "S_Port", "Seq"]

def label_dataset():

    print("Avvio etichettatura dataset")

    for file_path in (OFFICIAL_DATASET_PATH, ATTACK_DATASET_PATH):

        if not os.path.exists(file_path):

            raise FileNotFoundError(f"File non trovato: {file_path}")

    print(f"Caricamento dataset ufficiale da '{OFFICIAL_DATASET_PATH}'...")
    official_dataframe = pd.read_csv(OFFICIAL_DATASET_PATH)
    print(f"    {len(official_dataframe):,} pacchetti totali catturati")

    print(f"Caricamento dataset di attacco da '{ATTACK_DATASET_PATH}'...")
    attack_dataframe = pd.read_csv(ATTACK_DATASET_PATH)
    print(f"    {len(attack_dataframe):,} pacchetti di attacco inviati")

    for key in MATCHING_KEYS:

        if key not in official_dataframe.columns:

            raise ValueError(f"Colonna '{key}' mancante in {OFFICIAL_DATASET_PATH}")
        
        if key not in attack_dataframe.columns:

            raise ValueError(f"Colonna '{key}' mancante in {ATTACK_DATASET_PATH}")

    attack_fingerprint = set(

        map(tuple, attack_dataframe[MATCHING_KEYS].itertuples(index=False, name=None))
    )

    print(f"Creati {len(attack_fingerprint):,} fingerprint univoci di attacco")

    official_tuples = official_dataframe[MATCHING_KEYS].itertuples(index=False, name=None)
    official_dataframe["Label"] = [

        1 if t in attack_fingerprint else 0
        for t in official_tuples
    ]

    attack_count = int(official_dataframe["Label"].sum())
    normal_count = len(official_dataframe) - attack_count
    attack_percentage = 100.0 * attack_count / len(official_dataframe)

    print("\nRisultati etichettatura:")
    print(f"  Attacco  (Label=1):  {attack_count:>6,}  ({attack_percentage:5.2f}%)")
    print(f"  Normale  (Label=0):  {normal_count:>6,}  ({100.0 - attack_percentage:5.2f}%)")
    print(f"  Totale               {len(official_dataframe):>6,}")

    missing_attack_count = len(attack_fingerprint) - attack_count

    if missing_attack_count > 0:

        missing_percentage = 100.0 * missing_attack_count / len(attack_fingerprint)
        print(f"Pacchetti di attacco inviati ma non catturati: "
              f"{missing_attack_count:,} ({missing_percentage:.2f}%)")
        print("    (probabilmente bloccati dal firewall o persi in transito)")

    official_dataframe.to_csv(OUTPUT_DATASET_PATH, index=False)
    print(f"Dataset etichettato salvato in: '{OUTPUT_DATASET_PATH}'")

    print("\nAnteprima del dataset (prime 5 righe):")
    print(official_dataframe.head())

    print("\nEsempio di pacchetti classificati come attacco (se presenti):")
    attack_samples = official_dataframe[official_dataframe["Label"] == 1].head(3)

    if len(attack_samples) > 0:

        print(attack_samples)

    else:

        print("(nessun pacchetto di attacco presente)")

    return official_dataframe

if __name__ == "__main__":
    
    label_dataset()

Avvio etichettatura dataset
Caricamento dataset ufficiale da 'dataset_tesi_integrale.csv'...
    74,935 pacchetti totali catturati
Caricamento dataset di attacco da 'dataset_attacco.csv'...
    21,105 pacchetti di attacco inviati
Creati 21,105 fingerprint univoci di attacco

Risultati etichettatura:
  Attacco  (Label=1):  21,105  (28.16%)
  Normale  (Label=0):  53,830  (71.84%)
  Totale               74,935
Dataset etichettato salvato in: 'dataset_tesi_etichettato.csv'

Anteprima del dataset (prime 5 righe):
           Time     Src_IP     Dst_IP  S_Port  D_Port  Flags  Len    Win  TTL  \
0  18:44:07.809  10.0.2.13  10.0.3.10   37264      80      2   54  64240   62   
1  18:44:07.821  10.0.3.10  10.0.2.13      80   37264     18   58  64240   63   
2  18:44:07.822  10.0.2.13  10.0.3.10   37264      80      4   54      0   62   
3  18:44:07.892  10.0.2.13  10.0.3.10   37264      80     24  151  64240   62   
4  18:44:07.892  10.0.3.10  10.0.2.13      80   37264      4   54      0   63   
